In [1]:
import numpy as np
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:60], y[:60]

# Load and shuffle the dataset
X, y = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Apply PCA for feature reduction
n_components = 2  # Number of principal components to keep
pca = PCA(n_components=n_components)

# Fit PCA on training data and transform both training and test data
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Set dimensions for hyperdimensional space
D = n_components  # Adjust dimension according to PCA components

# Create a feature map
feature_map = PauliFeatureMap(feature_dimension=D, reps=3, entanglement='full')

sampler = Sampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

# Measure training time
start_train_time = time.time()

# Train a QSVM classifier with the PCA-transformed data
qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_pca, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_pca)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()

# Classify test data using the trained QSVM classifier
y_pred = qsvc.predict(X_test_pca)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum QSVM + PCA Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = sys.getsizeof(qsvc) + pca.n_components_ * (X_train.shape[1] + 1) * 8

print("Hybrid Classical-Quantum QSVM + PCA Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum QSVM + PCA Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum QSVM + PCA Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

C:\Users\C00591145\AppData\Local\Temp\ipykernel_1840\1810237198.py:42: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 77.50%
Hybrid Classical-Quantum QSVM + PCA Test Accuracy: 80.00%
Hybrid Classical-Quantum QSVM + PCA Training Time: 1.9524 seconds
Hybrid Classical-Quantum QSVM + PCA Inference Time: 2.0412 seconds
Hybrid Classical-Quantum QSVM + PCA Model Memory Required: 0.0005 MB


In [3]:
import numpy as np
import sys
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV1 as Sampler, QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Load and shuffle the dataset
X, y = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Apply PCA for feature reduction
n_components = 2  # Number of principal components to keep
pca = PCA(n_components=n_components)

# Fit PCA on training data and transform both training and test data
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Define Quantum Circuit with a higher number of reps
qc = QNNCircuit(ansatz=RealAmplitudes(n_components, reps=3))  # Increase reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

aer_sim = AerSimulator()
sampler = Sampler(backend=aer_sim)

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
    sampler=sampler
)

# Use L-BFGS-B optimizer for better performance with more complex models
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_pca, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_pca)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum QNN+PCA Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_pca)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum QNN+PCA Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = pca.n_components_ * (X_train.shape[1] + 1) * 8  # Approximate size of PCA components
model_memory += sys.getsizeof(sampler_classifier)

print("Hybrid Classical-Quantum QNN+PCA Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum QNN+PCA Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum QNN+PCA Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

C:\Users\C00591145\AppData\Local\Temp\ipykernel_1840\734578956.py:55: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=aer_sim)


Hybrid Classical-Quantum QNN+PCA Train Accuracy: 70.59%
Hybrid Classical-Quantum QNN+PCA Test Accuracy: 70.59%
Hybrid Classical-Quantum QNN+PCA Training Time: 53.1210 seconds
Hybrid Classical-Quantum QNN+PCA Inference Time: 0.1771 seconds
Hybrid Classical-Quantum QNN+PCA Model Memory Required: 0.0005 MB
